# История, окна и повторяемая публикация
MySQL задачи Stepik и локальная SQLite используют здесь совместимый поднабор.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "data").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "data").is_dir(), "Откройте блокнот из папки комплекта"
sys.path.insert(0, str(ROOT))
import pandas as pd
from analysis import basic, independent_basic, normalized, export_basic

import sqlite3
import pipeline
con=sqlite3.connect(":memory:")
con.executescript((ROOT / "sql/schema.sql").read_text(encoding="utf-8"))

In [2]:
query="""SELECT order_id,customer_id,SUM(amount_cents) OVER(PARTITION BY customer_id ORDER BY created_at,order_id ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS running FROM orders WHERE status='paid' AND currency='RUB'"""
display(pd.read_sql_query(query,con))

,order_id,customer_id,running
0,1,1,1000.0
1,2,1,3000.0
2,3,2,0.0
3,6,4,NaN


In [3]:
query="""SELECT i.item_id,p.price_cents,i.qty*p.price_cents AS item_total FROM items i LEFT JOIN prices p ON i.sku=p.sku AND i.bought_at>=p.valid_from AND (i.bought_at<p.valid_to OR p.valid_to IS NULL)"""
display(pd.read_sql_query(query,con))

,item_id,price_cents,item_total
0,1,100,200
1,2,120,120
2,3,80,240


In [4]:
s=pd.Series([10,20,30,40])
print(s.shift(1).rolling(2,min_periods=2).mean())

0     NaN
1     NaN
2    15.0
3    25.0
dtype: float64


## Реальные пакеты
Эти клетки обновляют outputs этого комплекта. Источники не изменяются. Ошибочные пакеты исследуются отдельно в pytest/acceptance.

In [5]:
first=pipeline.build(ROOT / "data/batches/first.csv",ROOT)
repeat=pipeline.build(ROOT / "data/batches/first.csv",ROOT)
assert first==repeat
print("REPEAT:",first["version"],first["rows"])

REPEAT: 115a2431cef675f0 4


In [6]:
before=(ROOT / "outputs/CURRENT.json").read_bytes()
try:
    pipeline.build(ROOT / "data/batches/late.csv",ROOT,fail="before-pointer")
except RuntimeError as error:
    print(error)
assert (ROOT / "outputs/CURRENT.json").read_bytes()==before
print("OLD VERSION RETAINED")

Injected failure before CURRENT: previous publication retained
OLD VERSION RETAINED


In [7]:
late=pipeline.build(ROOT / "data/batches/late.csv",ROOT)
print(pipeline.check(ROOT))
display(pipeline.read_current(ROOT)[1])

{'passed': True, 'version': '0c9a540cd43a8553', 'rows': 6, 'sources': 2}


,event_id,user_id,event_type,event_at,loaded_at
0,1,1,view,2026-01-01T10:00:00Z,2026-01-01T10:00:01Z
1,2,1,buy,2026-01-01T11:00:00Z,2026-01-01T11:00:01Z
2,3,2,buy,2026-01-02T10:00:00Z,2026-01-02T10:00:01Z
3,4,2,view,2026-01-02T11:00:00Z,2026-01-02T11:00:01Z
4,5,3,view,2026-01-02T12:00:00Z,2026-01-04T12:00:00Z
5,6,1,view,2026-01-03T10:00:00Z,2026-01-03T10:00:01Z


In [8]:
from analysis import funnel
events=pd.read_csv(ROOT / "data/events.csv",dtype="string")
print(funnel(events,"2026-01-10T00:00:00Z"))

{'eligible': 3, 'converted': 1, 'rate': 0.3333333333333333, 'users': ['1']}


## Ваша очередь
Повторите на отдельной распакованной копии с новым late-пакетом. Проверьте конфликт, отказ, сохранённую версию и возврат. Для воронки создайте новое событие ровно на правой границе семи суток: оно не должно войти.